In [2]:
# ============================================================
# CELL 1 — IMPORTS AND PATH SETUP
# ============================================================

import os
import json
import pandas as pd

# ------------------------------------------------------------
# Base directory
# ------------------------------------------------------------

BASE_DIR = os.getcwd()

# ------------------------------------------------------------
# Input dataset
# ------------------------------------------------------------

DATA_PATH = "datset\\model2_dummy.json"
# ------------------------------------------------------------
# LightGBM preprocessing output
# ------------------------------------------------------------

PREPROCESSED_DIR = os.path.join(
    BASE_DIR,
    "model2_preprocessed_lightgbm"
)

DATASET_OUTPUT_PATH = os.path.join(
    PREPROCESSED_DIR,
    "dataset.csv"
)

LABEL_MAPPING_PATH = os.path.join(
    PREPROCESSED_DIR,
    "label_mapping.json"
)

# Create output directory
os.makedirs(
    PREPROCESSED_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# Print configuration
# ------------------------------------------------------------

print("=" * 60)
print("MODEL 2 — LIGHTGBM PREPROCESSING")
print("=" * 60)

print("\nBase directory:")
print(BASE_DIR)

print("\nInput dataset:")
print(DATA_PATH)

print("\nPreprocessed output:")
print(PREPROCESSED_DIR)

print("\nDataset output:")
print(DATASET_OUTPUT_PATH)

print("\nLabel mapping:")
print(LABEL_MAPPING_PATH)

MODEL 2 — LIGHTGBM PREPROCESSING

Base directory:
c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\model_2(Medical Dialogue Manager)

Input dataset:
datset\model2_dummy.json

Preprocessed output:
c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\model_2(Medical Dialogue Manager)\model2_preprocessed_lightgbm

Dataset output:
c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\model_2(Medical Dialogue Manager)\model2_preprocessed_lightgbm\dataset.csv

Label mapping:
c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\model_2(Medical Dialogue Manager)\model2_preprocessed_lightgbm\label_mapping.json


In [3]:
# ============================================================
# CELL 2 — LOAD AND VALIDATE DATASET
# ============================================================

# Load JSON dataset
with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("=" * 60)
print("DATASET LOADED")
print("=" * 60)

print("\nTotal records:", len(data))

# ------------------------------------------------------------
# Basic structure validation
# ------------------------------------------------------------

required_fields = [
    "id",
    "context",
    "target",
    "safety"
]

errors = []

for i, record in enumerate(data):

    # Check top-level fields
    for field in required_fields:
        if field not in record:
            errors.append(
                f"Record {i}: missing field '{field}'"
            )

    # Check context
    if "context" in record:
        if not isinstance(record["context"], list):
            errors.append(
                f"Record {i}: context is not a list"
            )

        else:
            for j, message in enumerate(record["context"]):
                if not isinstance(message, dict):
                    errors.append(
                        f"Record {i}, message {j}: "
                        "not a dictionary"
                    )
                    continue

                if "role" not in message:
                    errors.append(
                        f"Record {i}, message {j}: "
                        "missing role"
                    )

                if "content" not in message:
                    errors.append(
                        f"Record {i}, message {j}: "
                        "missing content"
                    )

    # Check safety
    if "safety" in record:
        if not isinstance(record["safety"], dict):
            errors.append(
                f"Record {i}: safety is not a dictionary"
            )

        elif "red_flag" not in record["safety"]:
            errors.append(
                f"Record {i}: missing safety.red_flag"
            )


# ------------------------------------------------------------
# Validation result
# ------------------------------------------------------------

print("\nValidation errors:", len(errors))

if errors:
    print("\nFirst errors:")
    for error in errors[:10]:
        print("-", error)
else:
    print("Dataset structure is valid!")

# ------------------------------------------------------------
# Show one complete example
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SAMPLE RECORD")
print("=" * 60)

print(json.dumps(data[0], indent=2, ensure_ascii=False))

DATASET LOADED

Total records: 100

Validation errors: 0
Dataset structure is valid!

SAMPLE RECORD
{
  "id": "m2_0001",
  "context": [
    {
      "role": "user",
      "content": "I've had stomach pain since yesterday."
    }
  ],
  "target": "pain_location",
  "safety": {
    "red_flag": false
  }
}


In [4]:
# ============================================================
# CELL 3 — FLATTEN DIALOGUE CONTEXT
# ============================================================

def flatten_context(context):
    """
    Convert the structured dialogue context into
    one speaker-tagged text string.
    """

    messages = []

    for message in context:

        role = message["role"].strip().lower()
        content = message["content"].strip()

        if role == "user":
            messages.append(f"User: {content}")

        elif role == "assistant":
            messages.append(f"Assistant: {content}")

        else:
            messages.append(f"{role.capitalize()}: {content}")

    return " | ".join(messages)


# ------------------------------------------------------------
# Create flattened records
# ------------------------------------------------------------

processed_data = []

for record in data:

    flattened_text = flatten_context(
        record["context"]
    )

    processed_data.append({
        "id": record["id"],
        "text": flattened_text,
        "target": record["target"],
        "red_flag": record["safety"]["red_flag"]
    })


# ------------------------------------------------------------
# Convert to DataFrame
# ------------------------------------------------------------

df = pd.DataFrame(processed_data)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 60)
print("DIALOGUE FLATTENING COMPLETE")
print("=" * 60)

print("\nTotal records:", len(df))

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 flattened conversations:")
print("-" * 60)

for i in range(min(5, len(df))):
    print(f"\nRecord {i + 1}")
    print("ID:", df.iloc[i]["id"])
    print("Text:", df.iloc[i]["text"])
    print("Target:", df.iloc[i]["target"])
    print("Red flag:", df.iloc[i]["red_flag"])

DIALOGUE FLATTENING COMPLETE

Total records: 100

Columns:
['id', 'text', 'target', 'red_flag']

First 5 flattened conversations:
------------------------------------------------------------

Record 1
ID: m2_0001
Text: User: I've had stomach pain since yesterday.
Target: pain_location
Red flag: False

Record 2
ID: m2_0002
Text: User: I've had stomach pain since yesterday. | Assistant: Where exactly is the pain? | User: Mostly around my upper abdomen.
Target: pain_quality
Red flag: False

Record 3
ID: m2_0003
Text: User: I've had stomach pain since yesterday. | Assistant: Where exactly is the pain? | User: Mostly around my upper abdomen. | Assistant: How would you describe the pain? | User: It's a dull ache.
Target: pain_severity
Red flag: False

Record 4
ID: m2_0004
Text: User: I've had stomach pain since yesterday. | Assistant: Where exactly is the pain? | User: Mostly around my upper abdomen. | Assistant: How severe is it from 0 to 10? | User: About 5.
Target: associated_symptoms
Red

In [6]:
# ============================================================
# CELL 4 — CREATE FINAL CLASSIFICATION LABEL
# ============================================================

def create_label(row):
    """
    Convert safety records into the RED_FLAG class.
    Keep normal target labels unchanged.
    """

    if row["red_flag"] is True:
        return "RED_FLAG"

    return row["target"]


# ------------------------------------------------------------
# Create final label column
# ------------------------------------------------------------

df["label"] = df.apply(
    create_label,
    axis=1
)


# ------------------------------------------------------------
# Validate labels
# ------------------------------------------------------------

print("=" * 60)
print("FINAL LABEL CREATION")
print("=" * 60)

print("\nTotal records:", len(df))

print("\nMissing labels:", df["label"].isna().sum())

print("\nUnique labels:", df["label"].nunique())

print("\nLabel distribution:")
print("-" * 60)

label_counts = df["label"].value_counts()

for label, count in label_counts.items():
    print(f"{label:30s} : {count}")


# ------------------------------------------------------------
# Check RED_FLAG conversion
# ------------------------------------------------------------

red_flag_rows = df[df["red_flag"] == True]

print("\n" + "=" * 60)
print("RED_FLAG CHECK")
print("=" * 60)

print("RED_FLAG records:", len(red_flag_rows))

print("\nLabels assigned to RED_FLAG records:")
print(red_flag_rows["label"].unique())

FINAL LABEL CREATION

Total records: 100

Missing labels: 0

Unique labels: 36

Label distribution:
------------------------------------------------------------
associated_symptoms            : 25
RED_FLAG                       : 15
functional_impact              : 8
pain_location                  : 7
pain_severity                  : 6
pain_quality                   : 5
injury_or_trigger              : 3
pain_progression               : 2
cough_character                : 2
pain_triggers                  : 1
pain_relieving_factors         : 1
cough_frequency                : 1
cough_progression              : 1
dizziness_onset                : 1
dizziness_frequency            : 1
dizziness_progression          : 1
dizziness_triggers             : 1
dizziness_type                 : 1
throat_severity                : 1
fever_temperature              : 1
fever_progression              : 1
nausea_onset                   : 1
vomiting                       : 1
vomiting_frequency             :

In [7]:
# ============================================================
# CELL 5 — LOAD SENTENCE TRANSFORMER
# ============================================================

from sentence_transformers import SentenceTransformer

# ------------------------------------------------------------
# Model used for dialogue embeddings
# ------------------------------------------------------------

EMBEDDING_MODEL_NAME = "all-mpnet-base-v2"

print("=" * 60)
print("LOADING SENTENCE TRANSFORMER")
print("=" * 60)

print("\nModel:", EMBEDDING_MODEL_NAME)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("\nSentence Transformer loaded successfully!")

# Check embedding dimension
test_embedding = embedding_model.encode(
    ["Test medical conversation."],
    convert_to_numpy=True
)

print("Embedding dimension:", test_embedding.shape[1])

c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0910 21:43:57.789000 34280 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


LOADING SENTENCE TRANSFORMER

Model: all-mpnet-base-v2


c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\astha\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [


Sentence Transformer loaded successfully!
Embedding dimension: 768


In [8]:
# ============================================================
# CELL 6 — GENERATE SENTENCE EMBEDDINGS
# ============================================================

print("=" * 60)
print("GENERATING SENTENCE EMBEDDINGS")
print("=" * 60)

# ------------------------------------------------------------
# Extract flattened conversation text
# ------------------------------------------------------------

texts = df["text"].tolist()

print("\nNumber of conversations:", len(texts))

# ------------------------------------------------------------
# Generate embeddings
# ------------------------------------------------------------

embeddings = embedding_model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# ------------------------------------------------------------
# Check embedding shape
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("EMBEDDINGS GENERATED")
print("=" * 60)

print("\nEmbedding shape:", embeddings.shape)
print("Expected shape:", (len(df), 768))

print("\nData type:", embeddings.dtype)

# ------------------------------------------------------------
# Show one example
# ------------------------------------------------------------

print("\nFirst conversation:")
print(df.iloc[0]["text"])

print("\nFirst embedding (first 10 values):")
print(embeddings[0][:10])

GENERATING SENTENCE EMBEDDINGS

Number of conversations: 100


Batches: 100%|██████████| 7/7 [00:00<00:00, 17.71it/s]


EMBEDDINGS GENERATED

Embedding shape: (100, 768)
Expected shape: (100, 768)

Data type: float32

First conversation:
User: I've had stomach pain since yesterday.

First embedding (first 10 values):
[ 0.02428408  0.00896405 -0.01460921 -0.02107832  0.0234773   0.00698682
  0.01483896  0.04598504  0.03124572  0.03425662]


In [9]:
# ============================================================
# CELL 7 — SAVE PREPROCESSED DATA
# ============================================================

import numpy as np

# ------------------------------------------------------------
# Create embeddings output path
# ------------------------------------------------------------

EMBEDDINGS_PATH = os.path.join(
    PREPROCESSED_DIR,
    "embeddings.npy"
)

# ------------------------------------------------------------
# Create numeric label mapping
# ------------------------------------------------------------

labels = sorted(df["label"].unique())

label2id = {
    label: idx
    for idx, label in enumerate(labels)
}

id2label = {
    str(idx): label
    for label, idx in label2id.items()
}

# Add numeric label to dataframe
df["label_id"] = df["label"].map(label2id)

# ------------------------------------------------------------
# Save embeddings
# ------------------------------------------------------------

np.save(
    EMBEDDINGS_PATH,
    embeddings
)

# ------------------------------------------------------------
# Save processed dataset
# ------------------------------------------------------------

df.to_csv(
    DATASET_OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

# ------------------------------------------------------------
# Save label mapping
# ------------------------------------------------------------

label_mapping = {
    "label2id": label2id,
    "id2label": id2label
}

with open(
    LABEL_MAPPING_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        label_mapping,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("=" * 60)
print("PREPROCESSED DATA SAVED")
print("=" * 60)

print("\nEmbeddings:")
print(EMBEDDINGS_PATH)

print("\nDataset:")
print(DATASET_OUTPUT_PATH)

print("\nLabel mapping:")
print(LABEL_MAPPING_PATH)

print("\nEmbedding shape:", embeddings.shape)
print("Number of labels:", len(labels))
print("Number of records:", len(df))

print("\nFirst 5 label mappings:")
for label in labels[:5]:
    print(f"{label} -> {label2id[label]}")

PREPROCESSED DATA SAVED

Embeddings:
c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\model_2(Medical Dialogue Manager)\model2_preprocessed_lightgbm\embeddings.npy

Dataset:
c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\model_2(Medical Dialogue Manager)\model2_preprocessed_lightgbm\dataset.csv

Label mapping:
c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\model_2(Medical Dialogue Manager)\model2_preprocessed_lightgbm\label_mapping.json

Embedding shape: (100, 768)
Number of labels: 36
Number of records: 100

First 5 label mappings:
RED_FLAG -> 0
associated_symptoms -> 1
breath_onset -> 2
breath_progression -> 3
breath_severity -> 4


In [10]:
# ============================================================
# CELL 8 — CALCULATE CLASS WEIGHTS
# ============================================================

from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ------------------------------------------------------------
# Get labels
# ------------------------------------------------------------

y = df["label_id"].values

classes = np.unique(y)

# ------------------------------------------------------------
# Calculate balanced class weights
# ------------------------------------------------------------

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y
)

# ------------------------------------------------------------
# Convert to dictionary
# ------------------------------------------------------------

class_weights = {
    int(class_id): float(weight)
    for class_id, weight
    in zip(classes, class_weights_array)
}

# ------------------------------------------------------------
# Display weights with label names
# ------------------------------------------------------------

print("=" * 60)
print("CLASS WEIGHTS")
print("=" * 60)

for class_id in classes:

    label = id2label[str(class_id)]
    count = int((y == class_id).sum())
    weight = class_weights[class_id]

    print(
        f"{label:30s} | "
        f"Examples: {count:2d} | "
        f"Weight: {weight:.4f}"
    )

# ------------------------------------------------------------
# Save class weights
# ------------------------------------------------------------

CLASS_WEIGHTS_PATH = os.path.join(
    PREPROCESSED_DIR,
    "class_weights.json"
)

class_weights_named = {
    id2label[str(class_id)]: weight
    for class_id, weight in class_weights.items()
}

with open(
    CLASS_WEIGHTS_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        class_weights_named,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n" + "=" * 60)
print("CLASS WEIGHTS SAVED")
print("=" * 60)

print("\nLocation:")
print(CLASS_WEIGHTS_PATH)

CLASS WEIGHTS
RED_FLAG                       | Examples: 15 | Weight: 0.1852
associated_symptoms            | Examples: 25 | Weight: 0.1111
breath_onset                   | Examples:  1 | Weight: 2.7778
breath_progression             | Examples:  1 | Weight: 2.7778
breath_severity                | Examples:  1 | Weight: 2.7778
cough_character                | Examples:  2 | Weight: 1.3889
cough_frequency                | Examples:  1 | Weight: 2.7778
cough_progression              | Examples:  1 | Weight: 2.7778
diarrhea_frequency             | Examples:  1 | Weight: 2.7778
dizziness_frequency            | Examples:  1 | Weight: 2.7778
dizziness_onset                | Examples:  1 | Weight: 2.7778
dizziness_progression          | Examples:  1 | Weight: 2.7778
dizziness_triggers             | Examples:  1 | Weight: 2.7778
dizziness_type                 | Examples:  1 | Weight: 2.7778
duration                       | Examples:  1 | Weight: 2.7778
fever_progression              | Examples